## Bond valuation and scenario P&L

Completed on 27 November 2026 as guided practice with synthetic data.

For a bond with payment amounts $C_i$ at times $t_i$ in years, the model price is the sum of the discounted cash flows.

$$
P = \sum_i C_i e^{-r(t_i)t_i},
$$

where $r(t_i)$ is the continuously compounded zero rate for that payment date, expressed as a decimal. The notebook linearly interpolates the quoted rates and shocks at each payment date. By linearity, adding the interpolated shock gives the same rate as shocking the quotes first and then interpolating.

For a position of $q$ signed units, the scenario P&L is $q(P_{\mathrm{shocked}} - P_{\mathrm{base}})$.

### Optional shock input

`bond_prices_calculator` defaults to `shock_basis=0`. In the base case, pandas adds that scalar zero to every rate in the Series. In the shocked case, the function receives a Series in basis points, and pandas matches each shock to its rate by cash-flow index. Both cases use `(zero_rates + shock_basis / 100) / 100`, so the function needs a single rate calculation for bonds with any number of payments. The first division converts basis points to percentage points, and the second converts the adjusted percentage quote to a decimal rate.

### Largest loss contributor

Under the specified scenario, **B03 contributes the largest loss (€26.62)** in the independent repricing. The zero rate at 2.5 years falls from 2.875% to 2.650%, raising the present value of B03’s largest payment and driving up its bond price. The book is short 50 units of B03, so the price increase produces a loss.

### Verification

B01 pays €100 in one year. Its base and shocked prices agree with the independent benchmarks $100e^{-0.025}$ and $100e^{-0.027}$. A separate rerun with the curve shocks set to zero produced zero P&L for each position and for the book.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data" / "05_bond_book_yield_curve_twist"
if not data_dir.is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from options_lab.bonds import bond_prices


positions = pd.read_csv(data_dir / "positions.csv")
curve = pd.read_csv(data_dir / "curve.csv")
cashflows = pd.read_csv(data_dir / "cashflows.csv")

print(positions)
print(curve)
print(cashflows)


  bond_id  signed_units
0     B03           -50
1     B01            80
2     B02          -120
   maturity_years  zero_rate_pct  shock_bps
0             0.5           2.30         30
1             1.0           2.50         20
2             2.0           2.75        -10
3             3.0           3.00        -35
  bond_id  payment_years  cashflow_eur
0     B01            1.0         100.0
1     B02            0.5           2.0
2     B02            1.0           2.0
3     B02            1.5         102.0
4     B03            0.5           2.5
5     B03            1.0           2.5
6     B03            1.5           2.5
7     B03            2.0           2.5
8     B03            2.5         102.5


In [2]:
prices = bond_prices(cashflows, curve)

# B01 pays EUR 100 in one year. The curve quotes 2.50% base and 2.70% shocked.
b01 = prices.set_index("bond_id").loc["B01"]
np.testing.assert_allclose(
    [b01["price"], b01["shock_price"]],
    [100 * np.exp(-0.025), 100 * np.exp(-0.027)],
    rtol=0,
    atol=1e-10,
)

# Reprice with zero shocks at every quoted maturity.
zero_shock_prices = bond_prices(cashflows, curve.assign(shock_bps=0.0))
assert (zero_shock_prices["shock_price"] == zero_shock_prices["price"]).all()

In [3]:
# Finds shock prices for 
bond_shock_prices_n_positions = (prices.merge(positions,
                                        how="left",
                                        on="bond_id",
                                        validate="one_to_one")
                                        .assign(PnL=lambda table: (table["shock_price"] - table["price"]) * table["signed_units"])
                                        )


# Switches collumn order
bond_shock_prices_n_positions = bond_shock_prices_n_positions[["bond_id", "signed_units", "price", "shock_price", "PnL"]]


print(bond_shock_prices_n_positions)
print(f"The total PnL is {bond_shock_prices_n_positions["PnL"].sum():.3f} euros.")

  bond_id  signed_units       price  shock_price        PnL
0     B01            80   97.530991    97.336124 -15.589364
1     B02          -120  101.989544   101.909164   9.645550
2     B03           -50  105.070718   105.603165 -26.622373
The total PnL is -32.566 euros.
